In [1]:
import os
import json
import random
import asyncio
from dataclasses import dataclass
from dotenv import load_dotenv
import aiohttp
from datasets import load_dataset
from sklearn.metrics import roc_auc_score, accuracy_score
from tqdm.asyncio import tqdm

from config import PROBE_QUESTIONS, REWRITER_MODEL, TARGET_MODEL, N_SAMPLES_PER_CLASS
from data.rewriter import rewrite_all


In [2]:
load_dotenv()
random.seed(0)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

In [3]:
@dataclass
class Sample:
    """A single sample for evaluation awareness testing."""
    id: str
    transcript: str
    is_eval: int  # 1 = evaluation, 0 = real
    source: str

## Dataset to json

In [4]:
@dataclass
class Sample:
    """A single sample for evaluation awareness testing."""
    id: str
    transcript: str
    is_eval: int  # 1 = evaluation, 0 = real
    source: str

In [5]:
def save_samples(dataset_name: str, samples: list[Sample], n_samples: int = N_SAMPLES_PER_CLASS, output_file: str = "prompt_data.json"):
    """Save dataset samples to a JSON file for documentation."""
    from datetime import datetime
    from dataclasses import asdict

    with open(output_file, 'r') as f:
        output = json.load(f)

    output[dataset_name] = [asdict(s) for s in samples]
    output['metadata']['timestamp'] = datetime.now().isoformat()

    with open(output_file, "w") as f:
        json.dump(output, f, indent=2)

def load_mmlu_for_save(n: int = N_SAMPLES_PER_CLASS) -> list[Sample]:
    """Load MMLU questions as evaluation samples."""
    dataset = load_dataset("cais/mmlu", "all", split="test", trust_remote_code=True)

    random.seed(0)
    indices = random.sample(range(len(dataset)), min(n, len(dataset)))

    samples = []
    for idx in indices:
        item = dataset[idx]
        choices = "\n".join([f"{chr(65+i)}. {c}" for i, c in enumerate(item["choices"])])
        transcript = f"Question: {item['question']}\n\nChoices:\n{choices}"

        samples.append(Sample(
            id=f"mmlu_{idx}",
            transcript=transcript,
            is_eval=1,
            source="mmlu"
        ))

    return samples

def load_sharegpt_for_save(n: int = N_SAMPLES_PER_CLASS) -> list[Sample]:
    """Load ShareGPT conversations as real-world samples."""
    dataset = load_dataset(
        "anon8231489123/ShareGPT_Vicuna_unfiltered",
        data_files="ShareGPT_V3_unfiltered_cleaned_split_no_imsorry.json",
        split="train"
    )

    random.seed(0)
    indices = random.sample(range(len(dataset)), min(n*2, len(dataset)))

    samples = []
    for idx in indices:
        item = dataset[idx]

        conversations = item.get("conversations", [])
        if not conversations:
            continue

        first_human_msg = None
        for msg in conversations:
            if msg.get("from") == "human":
                first_human_msg = msg.get("value", "")
                break

        if not first_human_msg or len(first_human_msg.strip()) < 10:
            continue

        samples.append(Sample(
            id=f"sharegpt_{idx}",
            transcript=first_human_msg,
            is_eval=0,
            source="sharegpt"
        ))

        if len(samples) >= n:
            break

    return samples


In [6]:
def rephrase_mmlu_for_save():
    with open('data/prompt_data.json', 'r') as f:
        mmlu_from_saved = json.load(f)['mmlu']
    
    return rewrite_all(mmlu_from_saved, data_type='mcq')

In [7]:
# Run ONCE

# save_samples('mmlu', samples=load_mmlu_for_save())
# save_samples('sharegpt', samples=load_sharegpt_for_save())
save_samples('mmlu_rephrased', samples=rephrase_mmlu_for_save())

AuthenticationError: Error code: 401 - {'error': {'message': 'User not found.', 'code': 401}}

In [ ]:
mmlu_from_saved[0]